# 🏭 RL Training for Perishable Inventory MDP (Improved Version)

This notebook trains a PPO agent with **reward shaping** and **proper hyperparameters** for reliable learning.

## Key Improvements Over Basic Training

1. **Reward Shaping**: Normalized rewards + bonuses for service level
2. **Better Initial State**: Start with inventory to enable exploration
3. **Curriculum Learning**: Shorter episodes first, then increase
4. **Tuned Hyperparameters**: Higher entropy, more exploration

## 1. Setup

In [ ]:
# Install dependencies
!pip install stable-baselines3[extra] gymnasium numpy scipy matplotlib tensorboard -q
print("✅ Dependencies installed")

In [ ]:
import sys
import os

# Check if on Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repo on Colab
    if not os.path.exists('repo'):
        !git clone https://github.com/your-username/Multi-Supplier-Perishable-Inventory.git repo
    %cd repo
else:
    # Local - go to parent directory if in notebooks folder
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List
import time

# RL imports
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

# Inventory imports
from inventory_sim.gym_env import PerishableInventoryGymEnv
from inventory_sim.env import create_simple_mdp
from inventory_sim.simulation import run_episode
from inventory_agents import TailoredBaseSurgePolicy, BaseStockPolicy, DoNothingPolicy

print("✅ All imports successful!")

## 2. Environment Configuration (Optimized for Learning)

In [ ]:
# Environment configuration - THESE MATTER FOR LEARNING!
ENV_CONFIG = {
    "shelf_life": 5,
    "num_suppliers": 2,
    "mean_demand": 10.0,
    "fast_lead_time": 1,
    "slow_lead_time": 3,
    "fast_cost": 2.0,
    "slow_cost": 1.0,
    "max_order_per_supplier": 25,  # Slightly reduced for easier action space
    "order_step": 5,               # Actions: 0, 5, 10, 15, 20, 25
    "max_episode_steps": 100,      # SHORTER episodes for faster learning!
    "normalize_obs": True,
    "normalize_reward": True,      # KEY: Normalize rewards!
    "reward_shaping": True,        # KEY: Use reward shaping!
}

# Initial inventory - start with good stock to enable exploration
INITIAL_INVENTORY = np.array([15.0, 15.0, 15.0, 15.0, 15.0])  # 75 units total

print("Environment Configuration:")
for k, v in ENV_CONFIG.items():
    print(f"  {k}: {v}")
print(f"  initial_inventory: {INITIAL_INVENTORY.tolist()} (total: {INITIAL_INVENTORY.sum()})")

In [ ]:
# Test the environment
test_env = PerishableInventoryGymEnv(
    **ENV_CONFIG,
    initial_inventory=INITIAL_INVENTORY.copy(),
    render_mode="ansi"
)

print(f"Observation Space: {test_env.observation_space}")
print(f"Action Space: {test_env.action_space}")
print(f"\nAction meanings (order_step={ENV_CONFIG['order_step']}):")
for i in range(test_env.action_space.nvec[0]):
    print(f"  Action {i} → Order {i * ENV_CONFIG['order_step']} units")

In [ ]:
# Test reward shaping
print("Testing reward shaping with different scenarios...\n")

obs, info = test_env.reset(seed=42)

# Test 1: Order nothing (should have negative reward due to stockouts)
action_nothing = np.array([0, 0])
obs, reward_nothing, _, _, info = test_env.step(action_nothing)
print(f"Action [0,0] (no order): reward = {reward_nothing:.3f}")

# Test 2: Order enough to meet demand
test_env.reset(seed=42)
action_order = np.array([0, 2])  # Order 10 from slow supplier
obs, reward_order, _, _, info = test_env.step(action_order)
print(f"Action [0,2] (order 10 from slow): reward = {reward_order:.3f}")

# Test 3: Order moderately
test_env.reset(seed=42)
action_moderate = np.array([1, 2])  # Order 5 fast + 10 slow
obs, reward_moderate, _, _, info = test_env.step(action_moderate)
print(f"Action [1,2] (order 5 fast + 10 slow): reward = {reward_moderate:.3f}")

print("\n✅ Rewards are in reasonable range (approx -3 to +1)")

## 3. Training Configuration (Tuned for This Problem)

In [ ]:
# TUNED hyperparameters for inventory control
TRAINING_CONFIG = {
    "total_timesteps": 200_000,    # More steps for better learning
    "n_envs": 8,                   # More parallel envs = more diverse experience
    "learning_rate": 1e-3,         # Higher LR for faster initial learning
    "n_steps": 512,                # Steps per update (shorter for this problem)
    "batch_size": 128,             # Larger batch for stability
    "n_epochs": 10,
    "gamma": 0.99,
    "ent_coef": 0.05,              # HIGHER entropy = more exploration (KEY!)
    "clip_range": 0.2,
    "gae_lambda": 0.95,
    "max_grad_norm": 0.5,
    "vf_coef": 0.5,
}

# Network architecture - larger network
POLICY_KWARGS = {
    "net_arch": dict(
        pi=[256, 256],  # Policy network
        vf=[256, 256]   # Value network
    )
}

os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)

print("Training Configuration:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Create environment factory
def make_env():
    env = PerishableInventoryGymEnv(
        **ENV_CONFIG,
        initial_inventory=INITIAL_INVENTORY.copy()
    )
    return env

# Create vectorized training environments
train_env = make_vec_env(make_env, n_envs=TRAINING_CONFIG["n_envs"])

# Create evaluation environment
eval_env = Monitor(make_env())

print(f"✅ Created {TRAINING_CONFIG['n_envs']} parallel training environments")

## 4. Create and Train PPO Agent

In [ ]:
# Custom callback to track fill rate during training
class FillRateCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.fill_rates = []
        
    def _on_step(self):
        # Log fill rate from info
        for info in self.locals.get('infos', []):
            if 'fill_rate' in info:
                self.fill_rates.append(info['fill_rate'])
        return True

In [ ]:
# Create PPO model
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=TRAINING_CONFIG["learning_rate"],
    n_steps=TRAINING_CONFIG["n_steps"],
    batch_size=TRAINING_CONFIG["batch_size"],
    n_epochs=TRAINING_CONFIG["n_epochs"],
    gamma=TRAINING_CONFIG["gamma"],
    ent_coef=TRAINING_CONFIG["ent_coef"],
    clip_range=TRAINING_CONFIG["clip_range"],
    gae_lambda=TRAINING_CONFIG["gae_lambda"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    vf_coef=TRAINING_CONFIG["vf_coef"],
    verbose=1,
    tensorboard_log="./logs/",
    policy_kwargs=POLICY_KWARGS,
)

print(f"✅ PPO model created")
print(f"\nPolicy architecture:")
print(model.policy)

In [ ]:
# Create callbacks
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./models/best/",
    log_path="./logs/eval/",
    eval_freq=5000,
    n_eval_episodes=5,
    deterministic=True,
)

fill_rate_callback = FillRateCallback()

print("✅ Callbacks created")

In [ ]:
# TRAIN!
print("🚀 Starting training...")
print(f"   Total timesteps: {TRAINING_CONFIG['total_timesteps']:,}")
print(f"   Estimated updates: {TRAINING_CONFIG['total_timesteps'] // (TRAINING_CONFIG['n_steps'] * TRAINING_CONFIG['n_envs'])}")
print("-" * 50)

start_time = time.time()

model.learn(
    total_timesteps=TRAINING_CONFIG["total_timesteps"],
    callback=[eval_callback, fill_rate_callback],
    progress_bar=True
)

training_time = time.time() - start_time
print("-" * 50)
print(f"✅ Training completed in {training_time:.1f}s ({training_time/60:.1f} min)")

In [ ]:
# Save final model
model.save("models/ppo_inventory_final")
print("💾 Model saved to models/ppo_inventory_final.zip")

## 5. Evaluate Trained Agent

In [ ]:
# Load best model if available
best_model_path = "models/best/best_model.zip"
if os.path.exists(best_model_path):
    model = PPO.load(best_model_path)
    print("📦 Loaded best model from evaluation")
else:
    print("📦 Using final trained model")

# Evaluate
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=20)
print(f"\n📊 RL Agent Performance (20 episodes):")
print(f"   Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")

In [ ]:
# Detailed evaluation with metrics
def evaluate_rl_agent(model, env, n_episodes=20):
    """Evaluate RL agent and compute detailed metrics."""
    all_rewards = []
    all_fill_rates = []
    all_costs = []
    
    for ep in range(n_episodes):
        obs, info = env.reset(seed=100 + ep)
        episode_reward = 0
        total_demand = 0
        total_sales = 0
        total_cost = 0
        
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            
            episode_reward += reward
            total_demand += info.get('demand', 0)
            total_sales += info.get('sales', 0)
            total_cost += info.get('period_cost', 0)
            
            if terminated or truncated:
                break
        
        all_rewards.append(episode_reward)
        all_fill_rates.append(total_sales / total_demand if total_demand > 0 else 1.0)
        all_costs.append(total_cost)
    
    return {
        "mean_reward": np.mean(all_rewards),
        "std_reward": np.std(all_rewards),
        "mean_fill_rate": np.mean(all_fill_rates),
        "std_fill_rate": np.std(all_fill_rates),
        "mean_cost": np.mean(all_costs),
    }

rl_results = evaluate_rl_agent(model, eval_env, n_episodes=20)
print("\n📊 Detailed RL Agent Metrics:")
print(f"   Mean Reward: {rl_results['mean_reward']:.2f} ± {rl_results['std_reward']:.2f}")
print(f"   Fill Rate: {rl_results['mean_fill_rate']:.2%} ± {rl_results['std_fill_rate']:.2%}")
print(f"   Avg Period Cost: {rl_results['mean_cost']:.2f}")

## 6. Compare with Baselines

In [ ]:
# Create MDP for baseline evaluation
mdp = create_simple_mdp(
    shelf_life=ENV_CONFIG["shelf_life"],
    num_suppliers=ENV_CONFIG["num_suppliers"],
    mean_demand=ENV_CONFIG["mean_demand"],
    fast_lead_time=ENV_CONFIG["fast_lead_time"],
    slow_lead_time=ENV_CONFIG["slow_lead_time"],
    fast_cost=ENV_CONFIG["fast_cost"],
    slow_cost=ENV_CONFIG["slow_cost"],
)

baselines = {
    "Do Nothing": DoNothingPolicy(),
    "Base Stock (S=60)": BaseStockPolicy(target_level=60.0, supplier_id=1),
    "TBS (50/25)": TailoredBaseSurgePolicy(
        slow_supplier_id=1, fast_supplier_id=0,
        base_stock_level=50.0, reorder_point=25.0
    ),
}

def evaluate_baseline(policy, mdp, n_episodes=20, n_periods=100):
    """Evaluate baseline policy."""
    all_fill_rates = []
    all_costs = []
    
    for ep in range(n_episodes):
        state = mdp.create_initial_state(initial_inventory=INITIAL_INVENTORY.copy())
        results, _ = run_episode(mdp, policy, num_periods=n_periods, seed=100+ep, initial_state=state)
        metrics = mdp.compute_inventory_metrics(results)
        
        all_fill_rates.append(metrics["fill_rate"])
        all_costs.append(metrics["average_cost"])
    
    return {
        "mean_fill_rate": np.mean(all_fill_rates),
        "std_fill_rate": np.std(all_fill_rates),
        "mean_cost": np.mean(all_costs),
    }

print("Evaluating baselines...\n")
baseline_results = {}
for name, policy in baselines.items():
    results = evaluate_baseline(policy, mdp, n_episodes=20, n_periods=ENV_CONFIG["max_episode_steps"])
    baseline_results[name] = results
    print(f"{name}:")
    print(f"   Fill Rate: {results['mean_fill_rate']:.2%}")
    print(f"   Avg Cost: {results['mean_cost']:.2f}")

In [ ]:
# Comparison summary
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
print(f"\n{'Policy':<25} {'Fill Rate':>12} {'Avg Cost':>12}")
print("-"*60)

for name, results in baseline_results.items():
    print(f"{name:<25} {results['mean_fill_rate']:>11.1%} {results['mean_cost']:>12.2f}")

print("-"*60)
print(f"{'RL Agent (PPO)':<25} {rl_results['mean_fill_rate']:>11.1%} {rl_results['mean_cost']:>12.2f}")
print("="*60)

# Check if RL agent beats TBS
tbs_fill = baseline_results['TBS (50/25)']['mean_fill_rate']
rl_fill = rl_results['mean_fill_rate']

if rl_fill >= 0.9:
    print("\n✅ RL agent achieves good service level!")
elif rl_fill >= 0.7:
    print("\n⚠️ RL agent shows improvement but needs more training.")
else:
    print("\n❌ RL agent needs more training. Try increasing total_timesteps.")

## 7. Visualize Agent Behavior

In [ ]:
# Run a sample episode and visualize
obs, info = eval_env.reset(seed=123)

episode_data = {
    'step': [], 'inventory_position': [], 'demand': [], 
    'sales': [], 'order_fast': [], 'order_slow': [], 'reward': []
}

for step in range(ENV_CONFIG["max_episode_steps"]):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    
    episode_data['step'].append(step)
    episode_data['inventory_position'].append(info.get('inventory_position', 0))
    episode_data['demand'].append(info.get('demand', 0))
    episode_data['sales'].append(info.get('sales', 0))
    episode_data['order_fast'].append(action[0] * ENV_CONFIG['order_step'])
    episode_data['order_slow'].append(action[1] * ENV_CONFIG['order_step'])
    episode_data['reward'].append(reward)
    
    if terminated or truncated:
        break

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: Inventory Position
ax1 = axes[0]
ax1.plot(episode_data['step'], episode_data['inventory_position'], 'b-', lw=2, label='Inventory Position')
ax1.fill_between(episode_data['step'], 0, episode_data['demand'], alpha=0.3, color='red', label='Demand')
ax1.axhline(y=50, color='green', linestyle='--', alpha=0.5, label='Target (approx)')
ax1.set_ylabel('Units')
ax1.set_title('RL Agent: Inventory Position and Demand')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: Orders
ax2 = axes[1]
width = 0.8
ax2.bar(episode_data['step'], episode_data['order_slow'], width, label='Slow Supplier', color='steelblue', alpha=0.7)
ax2.bar(episode_data['step'], episode_data['order_fast'], width, bottom=episode_data['order_slow'], label='Fast Supplier', color='crimson', alpha=0.7)
ax2.set_ylabel('Order Quantity')
ax2.set_title('RL Agent: Order Decisions')
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: Cumulative Reward
ax3 = axes[2]
cumulative = np.cumsum(episode_data['reward'])
ax3.plot(episode_data['step'], cumulative, 'g-', lw=2)
ax3.fill_between(episode_data['step'], 0, cumulative, alpha=0.3, color='green')
ax3.set_xlabel('Step')
ax3.set_ylabel('Cumulative Reward')
ax3.set_title('RL Agent: Cumulative Reward')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('rl_agent_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Episode Fill Rate: {sum(episode_data['sales'])/sum(episode_data['demand']):.1%}")
print(f"📊 Final Reward: {cumulative[-1]:.2f}")

## 8. Tips for Better Training

If the agent still underperforms:

1. **Train longer**: Increase `total_timesteps` to 500K or 1M
2. **More exploration**: Increase `ent_coef` to 0.1
3. **Learning rate schedule**: Use linear decay
4. **Curriculum learning**: Start with very short episodes (50 steps)
5. **Reward tuning**: Adjust shaping weights in `gym_env.py`

In [ ]:
# Save model for download
model.save("models/ppo_inventory_final")

if IN_COLAB:
    from google.colab import files
    files.download('models/ppo_inventory_final.zip')

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE")
print("="*60)
print(f"Model saved to: models/ppo_inventory_final.zip")
print(f"\nNext steps:")
print("  1. Use inference.py to test locally")
print("  2. If fill rate < 90%, train longer (500K+ steps)")
print("  3. Tune reward shaping weights if needed")